In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Note] failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 25. Quantization — Making Models Smaller and Lighter [CPU/GPU Benchmark ⑪]

> **Learning Goals**
> - Understand precision, memory, and speed tradeoffs among FP32, FP16, BF16, INT8, and INT4
> - Derive the quantization formula $q = \mathrm{round}(x/s) + z$
> - Compare GPTQ, AWQ, and GGUF quantization methods

## 25.1 Why Quantization?

LLMs are expensive in memory and compute:
- LLaMA-7B FP16: 14GB
- Additional KV Cache during inference
- Difficult to deploy on mobile/edge devices

Quantization helps:
- FP16 → INT8: half the memory, about 2x speed
- FP16 → INT4: one quarter the memory, about 3-4x speed

## 25.2 Floating-Point Formats

| Type | Sign | Exponent | Mantissa | Total |
|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | 32 bit |
| FP16 | 1 | 5 | 10 | 16 bit |
| BF16 | 1 | 8 | 7 | 16 bit |
| FP8 (E4M3) | 1 | 4 | 3 | 8 bit |

- FP16 has good precision but a narrow range, so overflow is a risk
- BF16 has the same range as FP32 but lower precision, making it suitable for LLM training


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# floating-point range comparison
print("floating-point by type range:")
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16), ('BF16', torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"  {name}: min={info.min:.2e}, max={info.max:.2e}, eps={info.eps:.2e}")

# FP16 overflow example
print("\nFP16 limitations:")
print(f"  2^15 = {float(torch.tensor(2**15, dtype=torch.float16))}")
print(f"  2^16 = {float(torch.tensor(2**16, dtype=torch.float16))}  # inf!")
print(f"  BF16 2^16 = {float(torch.tensor(2**16, dtype=torch.bfloat16))}  # valid")


## 25.3 Integer Quantization Basics

Convert FP32 values to INT8 (0-255 or -128-127):

**Symmetric quantization** (zero-point = 0):
$$q = \mathrm{round}(x / s), \quad s = \frac{\max|x|}{127}$$
$$\hat{x} = s \cdot q$$

**Asymmetric quantization** (with zero-point):
$$q = \mathrm{round}(x / s) + z$$
$$s = \frac{\max(x) - \min(x)}{255}, \quad z = -\mathrm{round}(\min(x) / s)$$

Quantization error: $\epsilon = x - \hat{x}$.


In [ ]:
# quantization implementation
def quantize_symmetric(x, n_bits=8):
    """Symmetric Quantization."""
    qmax = 2 ** (n_bits - 1) - 1  # 127 for INT8
    scale = x.abs().max() / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.to(torch.float32) * scale

# Test
torch.manual_seed(0)
x = torch.randn(100) * 5  # Mean 0, Standard Deviation 5
q, scale = quantize_symmetric(x, n_bits=8)
x_hat = dequantize_symmetric(q, scale)
error = (x - x_hat).abs()
print(f"original range: [{x.min():.3f}, {x.max():.3f}]")
print(f"Quantization range: [{q.min()}, {q.max()}]")
print(f"scale: {scale:.6f}")
print(f"Mean Absolute error: {error.mean():.6f}")
print(f"Maximum Absolute error: {error.max():.6f}")
print(f"relative Error: {(error.mean() / x.abs().mean() * 100):.3f}%")

# Compare different bit widths
print("\nby bit width Quantization Error:")
for n_bits in [8, 4, 2]:
    q, scale = quantize_symmetric(x, n_bits=n_bits)
    x_hat = dequantize_symmetric(q, scale)
    err = (x - x_hat).abs().mean()
    print(f"  INT{n_bits}: Mean Error = {err:.6f} ({err/x.abs().mean()*100:.2f}%)")


## 25.4 Per-Tensor, Per-Channel, Per-Group

- **Per-tensor**: one scale for the whole tensor. Simple but less accurate.
- **Per-channel**: one scale per channel (row/column). More accurate, slightly more metadata.
- **Per-group**: one scale per group of $g$ values. A practical balance and the LLM quantization standard.

Example: 4-bit quantization with group_size=128 stores one FP16 scale for every 128 values.


In [ ]:
# Per-group quantization
def quantize_per_group(x, n_bits=4, group_size=128):
    """per-group Quantization."""
    qmax = 2 ** (n_bits - 1) - 1
    # split into groups
    orig_shape = x.shape
    x_flat = x.reshape(-1, group_size)
    scales = x_flat.abs().max(dim=1, keepdim=True).values / qmax
    q = torch.round(x_flat / scales).clamp(-qmax - 1, qmax)
    return q, scales, orig_shape

def dequantize_per_group(q, scales, orig_shape):
    return (q * scales).reshape(orig_shape)

# Comparison: per-tensor vs per-group
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.1  # Weight Matrix

# per-tensor (4-bit)
qmax = 7  # 4-bit symmetric
scale_t = W.abs().max() / qmax
q_t = torch.round(W / scale_t).clamp(-8, 7)
W_hat_t = q_t * scale_t
err_t = (W - W_hat_t).abs().mean()

# per-group (4-bit, group=128)
q_g, scales_g, shape = quantize_per_group(W, n_bits=4, group_size=128)
W_hat_g = dequantize_per_group(q_g, scales_g, shape)
err_g = (W - W_hat_g).abs().mean()

print(f"4-bit quantization error:")
print(f"  Per-tensor: {err_t:.6f} ({err_t/W.abs().mean()*100:.2f}%)")
print(f"  Per-group (128): {err_g:.6f} ({err_g/W.abs().mean()*100:.2f}%)")
print("\n=> Per-group is much more accurate. LLM Quantization Standard.")


## 25.5 PTQ (Post-Training Quantization) vs QAT

**PTQ**: quantize after training. Simple, but accuracy may drop.

**QAT (Quantization-Aware Training)**: simulate quantization during training. More accurate, but more complex.

LLMs usually use PTQ because retraining is too expensive.

## 25.6 GPTQ, AWQ

### GPTQ
- Based on minimizing second-order error
- Quantizes layer by layer
- Uses an approximate Hessian to predict error
- Preserves PPL well even at INT4

### AWQ (Activation-aware Weight Quantization)
- Keeps some important weight channels in FP16
- Determines importance from activation magnitudes
- Faster and simpler than GPTQ


## 25.7 QLoRA — 4-bit Quantization + LoRA

QLoRA (Dettmers et al., 2023):
1. Quantize the base model to 4-bit NF4 and freeze it
2. Train only LoRA adapters in FP16/BF16
3. Enables fine-tuning a 7B model on a single 24GB GPU

NF4 (NormalFloat 4-bit) is a 4-bit reavailableation optimized for normally distributed weights.


In [ ]:
# QLoRA memory estimate
def qlora_memory(base_params_b, adapter_ratio=0.01):
    """QLoRA Memory Estimation (GB).
    base_params_b: base Parameter Count (unit: billion)
    adapter_ratio: adapter parameters Ratio
    """
    # base: 4-bit = 0.5 bytes
    base_mem = base_params_b * 0.5
    # adapter: FP16 + AdamW (4x) = 8 bytes
    adapter_mem = base_params_b * adapter_ratio * 8
    # activations (roughly)
    activation = 2  # GB
    return base_mem + adapter_mem + activation

# Comparison: full fine-tuning vs QLoRA
print(f"{'Model':>10} | {'Full FT FP16 (GB)':>15} | {'QLoRA 4-bit (GB)':>17}")
print("-" * 50)
for name, n_b in [('7B', 7), ('13B', 13), ('30B', 30), ('70B', 70)]:
    full = n_b * 2 * 4 + 4  # FP16 + AdamW FP32 (4x) + activations
    qlora = qlora_memory(n_b, 0.01)
    print(f"{name:>10} | {full:>15.1f} | {qlora:>17.1f}")
print("\n=> With QLoRA, even a 70B model can be fine-tuned on a single 48GB GPU.")


## 25.8 [CPU/GPU Benchmark ⑪] Inference by Precision


In [ ]:
# by precision inference speed comparison
from llm_math.bench import time_fn

# simple linear layer inference
def bench_linear(d_in, d_out, batch_size, dtype, device='cpu'):
    """LinearLayer Inference Time."""
    model = nn.Linear(d_in, d_out).to(dtype=dtype, device=device)
    x = torch.randn(batch_size, d_in, dtype=dtype, device=device)
    def run():
        with torch.no_grad():
            return model(x)
    return run

print(f"{'dtype':>10} | {'CPU (ms)':>10} | {'Memory (MB)':>12}")
print("-" * 40)
d_in, d_out, bs = 4096, 4096, 8
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16)]:
    if dtype == torch.float16 and not torch.cuda.is_available():
        # FP16 support is limited on CPU
        continue
    try:
        run = bench_linear(d_in, d_out, bs, dtype, 'cpu')
        # warmup
        run()
        res = time_fn(run, device='cpu', warmup=2, repeat=5)
        # Memory (roughly)
        mem = d_in * d_out * (4 if dtype == torch.float32 else 2) / 1024**2
        print(f"{name:>10} | {res['mean_ms']:>10.3f} | {mem:>12.2f}")
    except Exception as e:
        print(f"{name:>10} | {'N/A':>10} | error: {e}")

print("\n=> FP16 half the memory, speed improvement is limited on CPU (large improvement on GPU).")


## 25.9 Key Takeaways

| Method | Bits | Memory Saving | Accuracy Loss |
|---|---|---|---|
| FP16 | 16 | 2x | almost none |
| BF16 | 16 | 2x | almost none |
| INT8 | 8 | 4x | small |
| INT4 (GPTQ/AWQ) | 4 | 8x | small |
| QLoRA (NF4) | 4 | 8x | small |

## Exercises

1. Measure MSE in a linear layer after INT8 quantization.
2. Compare quantization error for per-tensor, per-channel, and per-group quantization.
3. Compare quantization error for group sizes 32, 64, 128, and 256.
4. Compare FP32 vs FP16 inference speed on CPU and GPU.
5. Compute QLoRA memory savings versus full fine-tuning for 7B, 13B, and 70B models.

> Solutions: `solutions/ch25_solutions.ipynb`
